# 최신지우unsup_search 원본피처/전처리 GMM＋지도head
정상GMMteacher는공식평가이전비라벨만학습. 지도head는기존시간블록중첩검증. 원본비지도전체불량행AP와구분.


In [1]:
from pathlib import Path
import ast,sys,importlib,json,nbformat
import numpy as np,pandas as pd
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
UP=ROOT/"tmp/jiwoo_unsup_f02c9fd/source/JIWOO"
sys.path.insert(0,str(UP/"final/src"));sys.path.insert(0,str(UP/"unsup_search"))
import final_model as fm
import models as peer_models
INTER=ROOT/"tmp/jiwoo_final_6ca6729_c0859f2aa953421a859d31dbc31e5e81/JIWOO/final/outputs/intermediate"
SCALES=json.loads((INTER/"base/reference_scales.json").read_text(encoding="utf-8"))
cs=(UP/"unsup_search/common.py").read_text(encoding="utf-8")
scope={"np":np,"pd":pd,"fm":fm,"raw":fm.raw,"ALL_RAW":fm.ALL_RAW,"PRODUCTS":("cn7","rg3"),
 "FEAT_C17":list(fm.FEATURES),"FEAT_RAW":[fm.raw(v) for v in fm.ALL_RAW],
 "REL":["rel_fill_speed","rel_plast","rel_hold","rel_sw_ratio","rel_rest"],
 "STEP_DEFAULT":{v:.01 for v in fm.ALL_RAW[:7]},"SCALES":SCALES,"EQUIP_W2":"650톤-우진2호기"}
names={"step_of","add_relations","ext_features","_prep_stream"}
tree=ast.parse(cs)
exec(compile(ast.Module(body=[v for v in tree.body if isinstance(v,ast.FunctionDef) and v.name in names],type_ignores=[]),str(UP/"unsup_search/common.py"),"exec"),scope)
cols=["dv_"+v for v in fm.ALL_RAW]+["dv_"+v for v in scope["REL"]]
assert len(cols)==26
raw=pd.read_csv(ROOT/"data/original_split/moldset_unlabeled_cn7.csv",parse_dates=["TimeStamp"])
raw=raw[raw.EQUIP_NAME.eq(scope["EQUIP_W2"])]
lab_ts=[];windows=[]
for product in ["cn7","rg3"]:
 b=pd.read_csv(INTER/"base"/(product+"_shots.csv"),parse_dates=["TimeStamp"])
 lab_ts.extend(b.TimeStamp.tolist())
 for lo,hi in b.groupby("run_id").TimeStamp.agg(["min","max"]).itertuples(index=False):
  windows.append((lo-pd.Timedelta(minutes=60),hi+pd.Timedelta(minutes=60)))
F=scope["_prep_stream"]((scope["EQUIP_W2"],"CN7"),raw,pd.Series(lab_ts),windows)
assert F is not None
ref=F[F.TimeStamp<pd.Timestamp("2020-10-16")].copy()
assert not ref.TimeStamp.isin(lab_ts).any()
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
from sklearn.preprocessing import RobustScaler
d=pd.read_csv(OUT/"cn7_reference_snapshot/cn7_163_gas_supervised_input_raw_shots.csv",parse_dates=["TimeStamp"]).sort_values(["run_id","TimeStamp"]).reset_index(drop=True)
y=d.gas.to_numpy(int);eligible=(d.gas|d.any_fail.eq(0)).to_numpy()
N=len(d);g=d.run_id.to_numpy();seq=d.seq.to_numpy();blocks=d.time_block.to_numpy()
size=d.groupby("run_id").TimeStamp.transform("size")
innerblocks=(d.run_id+"__I"+(d.seq*5//size).astype(int).astype(str).str.zfill(2)).to_numpy()
assert set(ref.TimeStamp).isdisjoint(d.TimeStamp)
# Exact dev26 part of ext_features, skipping unused c17 calculation for speed.
def ext26(a):
 a=a.sort_values(["run_id","TimeStamp"]).reset_index(drop=True).copy()
 for v in fm.ALL_RAW:
  a["dv_"+v]=fm._by_run(a,fm.raw(v),lambda x,s=scope["step_of"](v):fm.causal_robust_dev(x,s)).values
 a=scope["add_relations"](a)
 steps={"rel_fill_speed":1e-3,"rel_plast":1e-3,"rel_hold":.01,"rel_sw_ratio":1e-3,"rel_rest":.01}
 for v in scope["REL"]:
  a["dv_"+v]=fm._by_run(a,v,lambda x,s=steps[v]:fm.causal_robust_dev(x,s)).values
 return a
check=d.iloc[:60].copy()
a=scope["ext_features"](check);b=ext26(check)
assert np.allclose(a[cols],b[cols],equal_nan=True)
# Original full-covariance GMM2 and Prep, training only pre-event unlabeled reference.
prep=peer_models.Prep().fit(ref[cols].to_numpy(float))
teacher=peer_models.GMM(2,seed=0).fit(prep(ref[cols].to_numpy(float)))
reference_cut=float(np.quantile(teacher.score(prep(ref[cols].to_numpy(float))),.999))
reference_n=len(ref);reference_last=ref.TimeStamp.max()
print("SOURCE ALIGNED reference",reference_n,"all_clean",len(F),"strict past end",reference_last,flush=True)
featurememo={}
def features(ctx):
 key=tuple(np.flatnonzero(ctx))
 if key in featurememo:return featurememo[key]
 ii=np.flatnonzero(ctx);frame=d.iloc[ii].copy()
 seg=np.cumsum(np.r_[True,(g[ii][1:]!=g[ii][:-1])|(np.diff(ii)!=1)])
 frame["run_id"]=["SEG"+str(v).zfill(5) for v in seg]
 value=ext26(frame)[cols].to_numpy(float)
 full=np.full((N,26),np.nan);full[ii]=value
 featurememo[key]=full;return full
def train_context(parent,q):
 ctx=parent.copy()
 for run in pd.unique(g[q]):
  ids=q[g[q]==run];ctx[(g==run)&(seq>=seq[ids].min()-10)&(seq<=seq[ids].max()+10)]=False
 return ctx
def teacher_features(x):
 z=prep(x)
 return np.column_stack([teacher.score(z),teacher.m.predict_proba(z)])
MODELS=["reference_GMM","GMMscore_LR","RF26","GMM_LR_RFmean"]
def train_heads(x,yy):
 fit={"gas_count":int(yy.sum())}
 if not yy.any():fit["empty_gas"]=True;return fit
 mf=teacher_features(x);normal=yy==0
 scaler=RobustScaler().fit(mf[normal]);mx=np.clip(scaler.transform(mf),-20,20)
 fit["scale"]=scaler
 fit["LR"]=LogisticRegression(C=.1,class_weight="balanced",max_iter=2000,random_state=42).fit(mx,yy)
 fit["RF"]=RandomForestClassifier(n_estimators=60,max_depth=4,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2).fit(prep(x),yy)
 return fit
def scores(fit,x):
 mf=teacher_features(x)
 out={"reference_GMM":mf[:,0]}
 if fit.get("empty_gas"):
  out.update({v:np.zeros(len(x)) for v in MODELS if v!="reference_GMM"})
  return out
 out["GMMscore_LR"]=fit["LR"].predict_proba(np.clip(fit["scale"].transform(mf),-20,20))[:,1]
 out["RF26"]=fit["RF"].predict_proba(prep(x))[:,1]
 out["GMM_LR_RFmean"]=(out["GMMscore_LR"]+out["RF26"])/2
 return out
old=nbformat.read("notebooks/181_cn7_dev26_supervised_gmm.ipynb",as_version=4)
s="\n".join(c.source for c in old.cells if c.cell_type=="code")
runner=s[s.index("memo={};audit=[]"):s.index("# Deployment uses inner")]
runner=runner.replace("cn7_181","cn7_184")
runner=runner.replace("ready=np.isfinite(queryx).all(axis=1)","ready=np.ones(len(q),bool) # Original Prep explicitly fills missing dev features with0.")
runner=runner.replace("cut=cut_inner(y[ref],inner[n]);met=","cut=reference_cut if n=='reference_GMM' else cut_inner(y[ref],inner[n]);met=")
runner=runner.replace('opts.append((met["F1"],met["TP"],-met["FP"],n,cut))','if n!="reference_GMM": opts.append((met["F1"],met["TP"],-met["FP"],n,cut))')
exec(runner)
# Frozen reference baseline plus supervised-head deployment from full label crossfit.
parent=np.ones(N,bool);refids=np.flatnonzero(eligible)
inner={n:np.full(len(refids),np.nan) for n in MODELS}
for ib in pd.unique(innerblocks[refids]):
 mask=innerblocks[refids]==ib;ip,_,_,_=predict(parent,refids[mask])
 for n in MODELS:inner[n][mask]=ip[n]
opts=[]
thresholds={}
for n in MODELS:
 cut=reference_cut if n=="reference_GMM" else cut_inner(y[refids],inner[n])
 thresholds[n]=cut
 if n!="reference_GMM":
  m=stats(y[refids],inner[n]>=cut);opts.append((m["F1"],m["TP"],-m["FP"],n,cut))
chosen=max(opts)[3]
fullfit=train_heads(features(parent)[refids],y[refids])
us=scores(fullfit,F[cols].to_numpy(float))
ur=[]
for name in MODELS:
 pr=us[name]>=thresholds[name]
 F["prediction_"+name]=pr.astype(int)
 ur.append({"model":name,"deployment_choice":int(name==chosen),"clean_unlabeled":len(F),"alarms":int(pr.sum()),"alarm_percent":100*pr.mean(),"post_reference_n":int((F.TimeStamp>=pd.Timestamp("2020-10-16")).sum()),"post_reference_alarms":int((pr&(F.TimeStamp>=pd.Timestamp("2020-10-16"))).sum())})
um=pd.DataFrame(ur)
F[["TimeStamp","run_id"]+["prediction_"+n for n in MODELS]].to_csv(OUT/"cn7_184_source_aligned_unlabeled_predictions.csv",index=False)
um.to_csv(OUT/"cn7_184_source_aligned_unlabeled_metrics.csv",index=False)
print("\nLABEL METRICS\n",mm.to_string(index=False))
print("\nUNLABEL METRICS\n",um.to_string(index=False))
print("\nDEPLOY",chosen)
lines=["# 지우 최신unsup_search 원본 정렬 + dev26 지도 분류","",
 "origin/jiwoo f02c9fd를병합없이받아common.py/models.py/final/src를임시스냅샷으로읽었다.",
 "원본dev26의21+5/관계단위1e-3/기록단위JSON/causal_robust_dev/Prep/전체공분산GMM2를사용한다. 빠른dev26추출이원본ext_features와일치함을확인했다.",
 "우진2 CN7만정제하고라벨시각/라벨Run±60분/이상기록/짧은Run/배럴1<100 비가동을제거했다.",
 f"과거참조{reference_n}샷, 마지막{reference_last}. GMM teacher는이참조만학습하며공식가스평가시각은겹치지않는다.",
 "원본은비지도GMM음의로그밀도점수다. 우리는이를고정하고음의로그밀도+군집소속확률2개를라벨로학습한LR에입력하는지도classifier를추가했다.",
 "RF26과GMM-LR＋RF평균도비교했다.같은605샷/시간블록20/purge10/nested5 유지.",
 "지도모델선택·임계값은안쪽에서만계산했다.원본비지도기준99.9%는지도모델선택에서제외한별도비교다.",
 "원본보고서는전체불량행기준AP이므로우리가스샷기준F1과직접비교하면안된다.",
 "원본Prep가결측편차를0으로처리하므로엄격한26개모두유효필터를쓰지않았다. 비라벨정답없는경보비율만보고한다.",
 "같은Run시간블록은미래라벨학습자료도포함할수있으며새Run/미래독립검증이아니다.참조GMM학습시각만공식평가이전으로고정했다.","",
 "## 라벨",mm.to_string(index=False),"","## 정제비라벨",um.to_string(index=False)]
(ROOT/"reports/cn7_184_source_aligned_gmm_supervised.md").write_text("\n".join(lines),encoding="utf-8")


SOURCE ALIGNED reference 4288 all_clean 8007 strict past end 2020-10-15 23:26:18


Completed dev26 supervised GMM 1/20


Completed dev26 supervised GMM 2/20


Completed dev26 supervised GMM 3/20


Completed dev26 supervised GMM 4/20


...

Completed dev26 supervised GMM 18/20


Completed dev26 supervised GMM 19/20


Completed dev26 supervised GMM 20/20



LABEL METRICS
                  model   N  gas  TP  FP  FN  TN   F1  precision   recall       AP
         reference_GMM 605   13   2   1  11 591 0.25   0.666667 0.153846 0.282031
           GMMscore_LR 605   13   0   3  13 589 0.00   0.000000 0.000000 0.018649
                  RF26 605   13   0   0  13 592 0.00   0.000000 0.000000 0.017362
         GMM_LR_RFmean 605   13   0   1  13 591 0.00   0.000000 0.000000 0.013478
primary_inner_selected 605   13   0   3  13 589 0.00   0.000000 0.000000 0.016358

UNLABEL METRICS
         model  deployment_choice  clean_unlabeled  alarms  alarm_percent  post_reference_n  post_reference_alarms
reference_GMM                  0             8007      11       0.137380              3719                      6
  GMMscore_LR                  1             8007       1       0.012489              3719                      0
         RF26                  0             8007       0       0.000000              3719                      0
GMM_LR_RFmean     

1769